# DeepWeeds Day 2 — notebook thực nghiệm

Import notebook này vào Kaggle bằng URL GitHub, bật **GPU** và **Internet** trong Settings, rồi chạy các ô theo thứ tự. Ô đầu tự clone source. Ảnh được tải trực tiếp từ Zenodo vào `/kaggle/working`; không cần upload dataset. Chỉ chọn cấu hình bằng **val**; test chỉ dùng ở ô chung kết.


In [ ]:
from pathlib import Path
import json, sys, platform, subprocess
REPO_DIR = Path.cwd()
if not (REPO_DIR / "eval.py").exists():
    REPO_DIR = Path("/kaggle/working/K4-Track4-Day2-Deeplearning-Advance") if Path("/kaggle/working").exists() else REPO_DIR / "K4-Track4-Day2-Deeplearning-Advance"
if not (REPO_DIR / "eval.py").exists():
    subprocess.run(["git", "clone", "https://github.com/Nituv05/K4-Track4-Day2-Deeplearning-Advance.git", str(REPO_DIR)], check=True)
assert (REPO_DIR / "eval.py").exists(), REPO_DIR
%cd {REPO_DIR}
!python -m pip -q install -r submissions/deepweeds_lab/code/requirements.txt
import torch, timm, pandas as pd
print(platform.python_version(), torch.__version__, timm.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")

## 0. Tải và kiểm dữ liệu

Ảnh tải từ [Zenodo](https://zenodo.org/records/7939060); `prepare` kiểm MD5 `b7b30f96d466fba86016aa5a26606e0f`. `eda` in số ảnh từng tập/lớp, kiểm giao rỗng và 17.509 ảnh, rồi lưu biểu đồ phân bố và 3 ảnh mẫu mỗi lớp.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py prepare
!python submissions/deepweeds_lab/code/experiments.py eda
print((Path("submissions/deepweeds_lab") / "eda.json").read_text())

## 0b. Kiểm pipeline

Overfit một batch 8 ảnh bằng head đóng băng, kiểm Focal γ=0 ≈ CE và xem ảnh CutMix trong `curves/`.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py sanity

## 1. Năm backbone

B01–B05 dùng cùng fold, seed và công thức nền. Đọc `summary.json`/`results.xlsx` để so sánh macro-F1 val, tham số, GMAC và thời gian.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py backbones

## 2. Công thức huấn luyện

T01–T12 thay từng yếu tố của B01: khởi tạo, augmentation, loss, sampler, Mixup/CutMix và EMA. B01 có cùng cấu hình với T00.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py training
base=Path("submissions/deepweeds_lab/runs")
candidates=[]
for exp in ["B01","B02","B03","B05"]+[f"T{i:02d}" for i in range(1,13)]:
    p=base/exp/"seed0"/"summary.json"
    if p.exists():
        s=json.loads(p.read_text()); candidates.append((exp,s["val_macro_f1"],s["val_top1"]))
print(sorted(candidates,key=lambda row:row[1],reverse=True))
SOURCE_EXP=max(candidates,key=lambda row:row[1])[0]
print("Val-selected source:",SOURCE_EXP)

## 3. Phương pháp suy luận trên val

I00–I05 đo macro-F1, ECE và độ trễ p50/p95/p99 với warmup 10 và 100 lần đo. Chọn phương pháp dựa trên val và ngân sách độ trễ của bạn.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py inference --source {SOURCE_EXP}
inf=json.loads(Path("submissions/deepweeds_lab/inference_results.json").read_text())
for r in inf: print(r["method"],r["val_macro_f1"],r["val_ece"],r["latency"]["p95"])
# Ưu tiên macro-F1, sau đó ECE và p95 khi hòa. Thay đổi quy tắc trước khi xem test nếu cần ràng buộc robot.
METHOD=sorted(inf,key=lambda r:(-r["val_macro_f1"],r["val_ece"],r["latency"]["p95"]))[0]["method"]
print("Val-selected method:",METHOD)

## 4. Chung kết: 3 seed, test một lượt mỗi seed

Chỉ chạy sau khi `SOURCE_EXP` và `METHOD` đã chốt bằng val. F01 và T00 cùng seed 0, 1, 2. `F01uncal` dùng cùng logit nếu chọn I04.

In [ ]:
!python submissions/deepweeds_lab/code/experiments.py final --source {SOURCE_EXP} --method {METHOD}
!python submissions/deepweeds_lab/code/experiments.py export

## 5. Chấm bằng `eval.py` gốc

Kết quả `score` phải trùng với `results.xlsx`. Dùng `grade` cho phần I rubric.

In [ ]:
!python eval.py score --pred "submissions/deepweeds_lab/predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
!python eval.py score --pred "submissions/deepweeds_lab/predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00 --out eval_out
!python eval.py grade --final "submissions/deepweeds_lab/predictions/F01_seed*_test.csv" --baseline "submissions/deepweeds_lab/predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --out eval_out